# Logistic regression classifier for greedy failure

This notebook trains an interpretable baseline classifier to predict whether ratio-greedy will fail on a knapsack instance. Dynamic programming supplies the ground-truth labels during training, but the model receives only features that can be computed without knowing the optimal solution.

The target label is:
$$
\text{failure} =
\begin{cases}
1 & \text{if greedy value} < \text{optimal DP value} \\
0 & \text{if greedy value} = \text{optimal DP value}
\end{cases}
$$

A predicted failure means that the later hybrid solver should consider running dynamic programming instead of trusting the greedy result.

### Load the labeled instance families

In [1]:
import ast
import numpy as np
import pandas as pd

from knapsack_ml_experiment import split_dataset_by_seed

independent = pd.read_csv("../artifacts/independent_instances.csv")
other = pd.read_csv("../artifacts/other_family_instances.csv")

dataset = pd.concat([independent, other], ignore_index=True)

# Restore the item sequences that were serialized as strings in the CSV files.
dataset["weights"] = dataset["weights"].apply(ast.literal_eval)
dataset["values"] = dataset["values"].apply(ast.literal_eval)

dataset.groupby("family")["failure"].agg(["count", "mean"])

### Represent each instance with numerical features

Logistic regression requires a fixed-length numerical row rather than a variable-length item list. These features summarize the size of the instance, the weight and value distributions, the value-to-weight ratios used by greedy, and the relationship between weights and values.

The model inputs deliberately exclude `optimal_value`, `relative_gap`, and `failure` because they are known only after running dynamic programming. Including them would leak the answer into the predictors. The `family` column is retained for later analysis but is not given to the model.

In [ ]:
def extract_features(row):
    weights = np.array(row["weights"])
    values = np.array(row["values"])
    ratios = values / weights

    return {
        "number_of_items": len(weights),
        "capacity_ratio": row["capacity"] / weights.sum(),
        "weight_mean": weights.mean(),
        "weight_std": weights.std(),
        "value_mean": values.mean(),
        "value_std": values.std(),
        "ratio_mean": ratios.mean(),
        "ratio_std": ratios.std(),
        "weight_value_correlation": np.corrcoef(weights, values)[0, 1],
    }

features = dataset.apply(extract_features, axis=1, result_type="expand")

# Keep the seed and family for splitting and analysis
model_data = pd.concat(
    [dataset[["seed", "family", "failure"]], features],
    axis=1,
)
model_data.head()

### Create seed-safe training, validation, and test sets

In [3]:
train, validation, test = split_dataset_by_seed(
    model_data,
    random_seed=0,
)

feature_columns = features.columns.tolist()

X_train = train[feature_columns]
y_train = train["failure"].astype(int)

# The test partition is intentionally not prepared or evaluated yet.
X_validation = validation[feature_columns]
y_validation = validation["failure"].astype(int)

{
    "training_rows": len(train),
    "validation_rows": len(validation),
    "test_rows": len(test),
}

### Train the logistic-regression baseline

`StandardScaler` centers and rescales every feature using statistics learned from the training set only. This is required because features (e.g. mean value and correlation) have different numerical scales. The pipeline applies the same learned transformation to validation data before logistic regression predicts whether each instance is a greedy failure.

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# The pipeline learns scaling parameters from X_train during fit.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)

model.fit(X_train, y_train)
validation_predictions = model.predict(X_validation)

/Users/assanbayg/Development/repos/knapsack-ml-experiment/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/assanbayg/Development/repos/knapsack-ml-experiment/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/assanbayg/Development/repos/knapsack-ml-experiment/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/assanbayg/Development/repos/knapsack-ml-experiment/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/assanbayg/Development/repos/knapsack-ml-experiment/.

### Evaluate failure detection

Class `1` represents a greedy failure. Its recall is the proportion of actual failures detected by the classifier. This matters more than accuracy alone because a false negative tells the future hybrid solver to trust greedy on an instance where greedy is suboptimal.

The confusion matrix is arranged as `[[true negatives, false positives], [false negatives, true positives]]`. False positives spend extra computation by unnecessarily selecting DP; false negatives can reduce solution quality.

In [6]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_validation, validation_predictions))
print(confusion_matrix(y_validation, validation_predictions))

              precision    recall  f1-score   support

           0       0.55      0.31      0.40       156
           1       0.79      0.91      0.85       444

    accuracy                           0.76       600
   macro avg       0.67      0.61      0.62       600
weighted avg       0.73      0.76      0.73       600

[[ 49 107]
 [ 40 404]]


#### Validation interpretation

At the default probability threshold of `0.5`, the classifier detects 404 of 444 greedy failures, giving failure recall of about 91%. It misses 40 failures and sends 107 successful-greedy instances to DP unnecessarily. The 76% accuracy is less informative on its own because failures are the majority class in this combined dataset.

This is only the first baseline. The next hybrid stage will vary the probability threshold and measure the tradeoff between missed failures and the percentage of instances sent to DP.

### Inspect the learned coefficients

Because the pipeline standardizes the inputs, coefficient magnitudes are reasonably comparable. A positive coefficient associates larger feature values with a higher predicted probability of greedy failure; a negative coefficient associates them with a lower probability. These are model associations, not proof that a feature causes failure.

In [5]:
coefficients = pd.Series(
    model.named_steps["logisticregression"].coef_[0],
    index=feature_columns,
).sort_values(key=abs, ascending=False)

coefficients

value_mean                  0.909130
value_std                  -0.750772
weight_value_correlation    0.561922
ratio_mean                 -0.294068
weight_mean                -0.203716
ratio_std                   0.100803
weight_std                  0.098450
capacity_ratio             -0.020354
number_of_items             0.000000
dtype: float64

### Learning checkpoint

The largest coefficients suggest which broad instance properties the linear model uses most strongly. A coefficient near zero can also occur when a feature does not vary: here every generated instance has 20 items, so `number_of_items` cannot help distinguish failures from successes. This will change when later distribution-shift experiments include larger instances.

Before using the test set, the remaining checks are to compare performance separately by family and decide how much failure recall the hybrid solver needs relative to its DP usage.